In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
from tqdm import tqdm
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import re

# Hyperparameters
d_model = 256
nhead = 8
num_layers = 4
dim_feedforward = 1024
max_seq_len = 256
batch_size = 32
learning_rate = 0.0005
epochs = 15
dropout = 0.1
top_k = 50
top_p = 0.85  # Adjusted for more coherence
temperature = 0.7

# Preprocess text to normalize punctuation while preserving newlines
def preprocess_text(text):
    # Replace multiple punctuation with single instances
    text = re.sub(r'[-]{2,}', '-', text)
    text = re.sub(r'([,.!;:\'\"\(\)])\1+', r'\1', text)
    text = re.sub(r'[^\w\s\n,.!;:\-\'\"\(\)]', '', text)  # Keep newlines (\n)
    text = re.sub(r'[ \t]+', ' ', text)  # Normalize spaces/tabs, preserve newlines
    return text.strip()

# Estimate average line length in tokens for post-processing
def estimate_line_length(text, vocab):
    lines = text.split('\n')
    token_lengths = [len(vocab.encode(line.strip())) for line in lines if line.strip()]
    return int(sum(token_lengths) / len(token_lengths)) if token_lengths else 10

# Vocabulary class with newline as special token
class Vocabulary:
    def __init__(self, text, vocab_size=10000):
        special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '\n', ',', '.', '!', ';', ':', '-', '(', ')', "'", '"']
        self.tokenizer = Tokenizer(BPE())
        self.tokenizer.pre_tokenizer = Whitespace()
        trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=special_tokens)
        self.tokenizer.train_from_iterator([text], trainer)
        self.vocab_size = self.tokenizer.get_vocab_size()
        self.pad_token_id = self.tokenizer.token_to_id('<PAD>')
        self.unk_token_id = self.tokenizer.token_to_id('<UNK>')
        self.bos_token_id = self.tokenizer.token_to_id('<BOS>')
        self.eos_token_id = self.tokenizer.token_to_id('<EOS>')
        self.newline_token_id = self.tokenizer.token_to_id('\n')
        self.punctuation_ids = [self.tokenizer.token_to_id(p) for p in [',', '.', '!', ';', ':', '-', '(', ')', "'", '"'] if self.tokenizer.token_to_id(p) is not None]

    def encode(self, text):
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True)

    def token_to_id(self, token):
        return self.tokenizer.token_to_id(token) or self.unk_token_id

    def id_to_token(self, id):
        return self.tokenizer.id_to_token(id) or '<UNK>'

# Sinusoidal Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Transformer Model with Encoder-Decoder
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, max_seq_len, dropout):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt):
        src = self.embedding(src) * math.sqrt(self.d_model)
        tgt = self.embedding(tgt) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        tgt = self.pos_encoder(tgt)
        output = self.transformer(src, tgt)
        output = self.fc(output)
        return output

# Data preparation
def prepare_data(text, vocab, seq_len):
    sequences = []
    encoded = vocab.encode(text)
    for i in range(0, len(encoded) - seq_len, 1):
        seq = encoded[i:i + seq_len]
        target = encoded[i + 1:i + seq_len + 1]
        sequences.append((seq, target))
    return sequences

# Training function
def train_model(model, data, vocab, epochs, device):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_token_id)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(range(0, len(data) - batch_size, batch_size), desc=f'Epoch {epoch+1}/{epochs}')
        for i in progress_bar:
            batch = data[i:i + batch_size]
            src = torch.tensor([seq for seq, _ in batch], dtype=torch.long).to(device)
            tgt = torch.tensor([tgt[:-1] for _, tgt in batch], dtype=torch.long).to(device)
            tgt_output = torch.tensor([tgt[1:] for _, tgt in batch], dtype=torch.long).to(device)

            optimizer.zero_grad()
            output = model(src, tgt)
            loss = criterion(output.view(-1, vocab.vocab_size), tgt_output.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})

        scheduler.step()
        avg_loss = total_loss / (len(data) // batch_size)
        print(f'Epoch {epoch+1}, Average Loss: {avg_loss:.4f}')

# Generation function with punctuation penalty and newline handling
def generate_text(model, seed_text, vocab, length, device, top_k, top_p, temperature, avg_line_length):
    model.eval()
    text = seed_text
    input_ids = torch.tensor([vocab.encode(seed_text[-max_seq_len:])], dtype=torch.long).to(device)
    tgt_ids = torch.tensor([vocab.encode(seed_text[-max_seq_len:])[:-1]], dtype=torch.long).to(device)
    recent_tokens = []
    token_count = 0

    with torch.no_grad():
        for _ in range(length):
            output = model(input_ids, tgt_ids)
            logits = output[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)

            # Punctuation penalty
            if len(recent_tokens) >= 3 and all(t in vocab.punctuation_ids for t in recent_tokens[-3:]):
                for pid in vocab.punctuation_ids:
                    probs[0, pid] *= 0.1

            # Top-p sampling
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = False
            probs[sorted_indices_to_remove] = 0
            probs = probs / probs.sum()

            # Top-k sampling
            top_k_probs, top_k_indices = torch.topk(probs, min(top_k, probs.size(-1)))
            top_k_probs = top_k_probs / top_k_probs.sum()
            next_token_idx = torch.multinomial(top_k_probs, 1).item()
            next_token_id = top_k_indices[0, next_token_idx].item()

            recent_tokens.append(next_token_id)
            if len(recent_tokens) > 5:
                recent_tokens.pop(0)

            decoded_token = vocab.decode([next_token_id])
            text += decoded_token
            token_count += 1

            # Insert newline based on average line length
            if token_count >= avg_line_length and next_token_id not in vocab.punctuation_ids:
                text += '\n'
                token_count = 0

            tgt_ids = torch.cat((tgt_ids[:, 1:], torch.tensor([[next_token_id]], dtype=torch.long).to(device)), dim=1)
            input_ids = torch.cat((input_ids[:, 1:], torch.tensor([[next_token_id]], dtype=torch.long).to(device)), dim=1)

    # Post-process to clean up punctuation and ensure newlines
    text = re.sub(r'([,.!;:\-\'\"\(\)])\1+', r'\1', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# Main execution
if __name__ == "__main__":
    # Read and preprocess input text file
    try:
        with open('author_text.txt', 'r', encoding='utf-8') as f:
            text = f.read()
    except FileNotFoundError:
        print("Error: 'author_text.txt' not found in the same directory as the script.")
        exit(1)
    except UnicodeDecodeError:
        print("Error: 'author_text.txt' must be encoded in UTF-8.")
        exit(1)

    text = preprocess_text(text)

    if len(text) < max_seq_len + 1:
        print(f"Error: Text file is too short. Must be at least {max_seq_len + 1} characters.")
        exit(1)

    # Build vocabulary
    vocab = Vocabulary(text, vocab_size=10000)
    avg_line_length = estimate_line_length(text, vocab)
    data = prepare_data(text, vocab, max_seq_len)

    if len(data) < batch_size:
        print(f"Error: Not enough data. Need at least {batch_size} sequences, but got {len(data)}.")
        exit(1)

    # Initialize model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TransformerModel(
        vocab_size=vocab.vocab_size,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        max_seq_len=max_seq_len,
        dropout=dropout
    )

    # Train model
    train_model(model, data, vocab, epochs, device)

    # Generate text
    seed_text = text[:max_seq_len]
    generated_text = generate_text(model, seed_text, vocab, length=750, device=device, top_k=top_k, top_p=top_p, temperature=temperature, avg_line_length=avg_line_length)
    print("Generated text:")
    print(generated_text)

Epoch 1/15:   0%|          | 1/959 [00:30<8:14:43, 30.98s/it, loss=8.69]


KeyboardInterrupt: 